In [ ]:
import torch
from unsloth import FastVisionModel 
from transformers import TextStreamer
import torch
from PIL import Image
import json
import re
import os
import pandas as pd

# Load the model and processor
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

def extract_json(text):
    match = re.search(r'\{[\s\S]*\}', text)
    if not match:
        raise ValueError("No JSON found")
    return json.loads(match.group())


def analyze_image_realism(image_path):
    """
    Analyze an image for realism using Qwen3-VL-8B-Instruct
    
    Args:
        image_path: Path to the image file or PIL.Image object
    
    Returns:
        dict: JSON response with realism analysis
    """
    # Load image if path is provided
    if isinstance(image_path, str):
        image = Image.open(image_path)
    else:
        image = image_path
    
    # Your prompt for realism analysis
    prompt = (
    "Is there anything unrealistic in this image? yes or no or somewhat, "
    "if yes or somewhat explain in maximum 30 words, please ensure to explain "
    "what looks unreal like if face is distorted, or transition between objects is not smooth."
    "Respond ONLY in the following JSON format:\n"
    "{\n"
    '  "unrealistic": "yes | no | somewhat",\n'
    '  "explanation": "string"\n'
    "}\n\n"
)
    # Prepare messages for the model
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt}
            ]
        }
    ]
    
    # Process inputs
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens = False,
        return_tensors = "pt",
    )
    inputs = inputs.to(model.device)
    
    output_text = model.generate(
        **inputs,
        # streamer = TextStreamer(tokenizer, skip_prompt = True),
        max_new_tokens = 200,
        use_cache = True,
        temperature = 1.5,
        min_p = 0.1,
    )
        
    generated_ids = output_text[0][inputs["input_ids"].shape[-1]:]

    output_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    output_text = extract_json(output_text)
    
    return output_text

# Example usage
if __name__ == "__main__":

    generated_descriptions = []
    PATH_TO_IMAGE_FOLDER = r"/home/manem/Qwen-3VL-Testing/dataset/images/test_images"
    PATH_TO_OUTPUT_CSV = r"/home/manem/Qwen-3VL-Testing/descriptions/before-finetuning/test"
    images = [os.path.join(PATH_TO_IMAGE_FOLDER, img) for img in os.listdir(PATH_TO_IMAGE_FOLDER) if img.endswith(('.png'))]
    for img_path in images:
        try:
            result = analyze_image_realism(img_path)
            label = result['unrealistic']
            explanation = result['explanation']

            img_path_formatted = os.path.basename(img_path)
            generated_descriptions.append({
                'image_path': img_path_formatted,
                'unrealistic': label,
                'explanation': explanation
            })
            print(generated_descriptions[-1])
        except Exception as e:
            print(f"Error processing {img_path}: {e}")
            generated_descriptions.append({
                'image_path': os.path.basename(img_path),
                'unrealistic': "error",
                'explanation': str(e)
            })
    
    # convert generated descriptions to csv and save
    df = pd.DataFrame(generated_descriptions)
    df.to_csv(os.path.join(PATH_TO_OUTPUT_CSV, "test_descriptions_realism_unsloth.csv"), index=False)

        

==((====))==  Unsloth 2026.1.4: Fast Qwen3_Vl patching. Transformers: 4.57.6.
   \\   /|    NVIDIA A40. Num GPUs = 1. Max memory: 44.339 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 2/2 [00:42<00:00, 21.45s/it]


{'image_path': 'f104.png', 'unrealistic': 'no', 'explanation': 'Image appears realistic with natural lighting, consistent perspective, and no visible distortions or unnatural transitions between objects or scenery.'}
{'image_path': 'f108.png', 'unrealistic': 'no', 'explanation': 'Image shows realistic aerial view of bridge and cityscape with natural lighting, reflections, and proportions. No distorted faces or unnatural transitions observed.'}
{'image_path': 'f114.png', 'unrealistic': 'no', 'explanation': 'Bald eagles appear naturally posed, lighting and textures consistent. No distorted faces or unnatural transitions visible in the image.'}
{'image_path': 'f124.png', 'unrealistic': 'no', 'explanation': "Bird's features and posture appear natural; lighting, texture, and focus realistic. No distorted face or unnatural object transitions detected."}
{'image_path': 'f126.png', 'unrealistic': 'no', 'explanation': 'Leaf and wood textures look natural, lighting and focus realistic. No distor

In [4]:
# I want to replace the descriptions of certain images with new descriptions generated by the model. I will read the existing CSV, update the descriptions, and save it back.
PATH_TO_IMAGE_FOLDER = r"/home/manem/Qwen-3VL-Testing/dataset/images/train_images"
PATH_TO_OUTPUT_CSV = r"/home/manem/Qwen-3VL-Testing/descriptions/before-finetuning/train"
CSV_FILE = os.path.join(PATH_TO_OUTPUT_CSV, "train_descriptions_realism_unsloth.csv")

# Read existing CSV
df = pd.read_csv(CSV_FILE)

# Filter rows where unrealistic == "error"
error_rows = df[df["unrealistic"] == "error"]

print(f"Found {len(error_rows)} images with error label")

for idx, row in error_rows.iterrows():
    img_name = row["image_path"]
    img_path = os.path.join(PATH_TO_IMAGE_FOLDER, img_name)

    try:
        result = analyze_image_realism(img_path)

        # Update dataframe
        df.at[idx, "unrealistic"] = result["unrealistic"]
        df.at[idx, "explanation"] = result["explanation"]

        print(f"Updated: {img_name}")

    except Exception as e:
        print(f"Still failing: {img_name} -> {e}")
        df.at[idx, "explanation"] = str(e)

# Save updated CSV
df.to_csv(CSV_FILE, index=False)

print("CSV updated successfully.")


Found 6 images with error label
Updated: f144.png
Updated: f166.png
Updated: f369.png
Updated: f467.png
Updated: r110.png
Updated: r55.png
CSV updated successfully.
